# Lab Assignment 3: Text Preprocessing, Tokenization, and Vector Representations

**Student:** Muhammad Obaidullah (abdulsalam.m@northeastern.edu)  
**Course:** Applied NLP  
**Submission:** Upload `Lab_3.ipynb` to Canvas with all cells run and outputs visible.

---

## Overview

Every NLP pipeline begins with decisions that are easy to underestimate: how do you split text into units? which surface variations do you collapse? what do you discard? These choices directly determine what information your model can and cannot see. A preprocessing step that improves sentiment classification may actively hurt named entity recognition. A tokenization strategy optimized for English morphology fails on agglutinative languages.

In this lab you will implement each of these transformations from scratch, measure their effects, and build the two foundational text representations — Bag of Words and TF-IDF — using only NumPy and Python. You will then compare your implementations against reference library output to verify correctness and explain where and why they differ.

---

## Learning Objectives

By the end of this lab you will be able to:

1. Evaluate different preprocessing strategies for a given task and justify which choices optimize downstream performance
2. Implement and compare tokenization strategies — word-level, subword (BPE), and WordPiece — and explain when each is appropriate
3. Apply normalization techniques (stemming, lemmatization) and evaluate their trade-offs for different tasks
4. Build BoW and TF-IDF representations from scratch and articulate what information each captures and loses

---

## Grading

| Component | Points | Method |
|---|---|---|
| Part 1 — Preprocessing pipeline | 20 | Autograded (14) + Manual (6) |
| Part 2 — Tokenization strategies | 20 | Autograded (14) + Manual (6) |
| Part 3 — Normalization | 18 | Autograded (12) + Manual (6) |
| Part 4 — BoW and TF-IDF from scratch | 27 | Autograded (21) + Manual (6) |
| Code quality | 15 | Manual |
| **Total** | **100** | |

### Manual grading rubric

**Reflections (6 pts each, Parts 1–4):**  
Each part has a written reflection prompt. Full credit requires: (a) a specific, justified claim — not "it depends" without explanation; (b) connection to a concrete downstream task; (c) use of precise terminology from J&M. Restating what the code does earns 1–2 pts. Explaining *why* a trade-off exists and *when* you would make each choice earns 5–6 pts.

**Code quality (15 pts):**

| Criterion | Points | What we look for |
|---|---|---|
| Docstrings and inline comments | 4 | Every function has a complete docstring. Non-obvious logic is commented. |
| Function design | 4 | Reusable; no hard-coded values that should be parameters; one job per function. |
| Defensive coding | 3 | Input validation where it matters; no silent failures. |
| Code clarity and naming | 2 | Descriptive names; reads like a narrative. |
| Avoiding redundancy | 2 | DRY; no copy-pasted logic. |

> **Note:** Code quality is now graded at full standard. Gaps that were flagged but not penalized in Labs 1–2 will be deducted here.

> **Docstring format:** This course uses **Google-style** docstrings. Every function you write must follow this format:
>
>     def example_function(count, label="item"):
>         """One-line summary of what the function does.
>
>         Args:
>             count (int): What this argument represents.
>             label (str): What this argument represents. Defaults to "item".
>
>         Returns:
>             list[str]: What the return value represents.
>         """
>
> Every code cell with a `TODO: Write your own docstring` placeholder expects a docstring in exactly this format — a one-line summary, then an `Args:` section (one bullet per parameter with its type), then a `Returns:` section with the return type. The markdown text immediately above each function gives you the argument names, types, and return type you need to describe.

---

## Setup

In [1]:
# Setup — run this cell first every session.

import subprocess
import sys


def install_if_missing(package):
    """Install a package if it cannot be imported.

    Args:
        package (str): The importable package name to check and install.

    Returns:
        None
    """
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", package, "-q"]
        )


for pkg in ["numpy", "pandas", "nltk", "scikit-learn", "tokenizers"]:
    install_if_missing(pkg)

import numpy as np
import string
import math
from collections import Counter, defaultdict

import nltk
for resource in ["punkt", "punkt_tab", "stopwords",
                 "wordnet", "omw-1.4", "averaged_perceptron_tagger"]:
    nltk.download(resource, quiet=True)

from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords

print("Environment ready.")

Environment ready.


---

## Working corpus

The same small corpus is used throughout the lab. Do not modify this cell — the autograder uses these exact strings.

`CORPUS_A` is the corpus embedded in the notebook (you can see it).  
`CORPUS_B` is a held-out corpus used by the autograder to verify your functions generalize.

In [2]:
# Do not modify this cell.

CORPUS_A = [
    "The dog barked at the mailman loudly",
    "The cat sat on the mat quietly",
    "Dogs and cats are common household pets",
    "The mailman delivers letters every morning",
    "A barking dog rarely bites",
]

# Stopword list used throughout this lab.
# Using a fixed set (rather than nltk.corpus.stopwords) keeps results
# deterministic across NLTK versions and makes your functions testable.
LAB_STOPWORDS = frozenset({
    'a', 'an', 'the', 'and', 'or', 'but', 'in', 'on', 'at',
    'to', 'for', 'of', 'with', 'by', 'are', 'is', 'was', 'be',
    'it', 'its', 'this', 'that', 'as', 'every', 'all',
})

print(f"Corpus A: {len(CORPUS_A)} documents")
print(f"Stopword list: {len(LAB_STOPWORDS)} words")

Corpus A: 5 documents
Stopword list: 25 words


---

## Part 1 — Preprocessing Pipeline (20 points)

Text preprocessing is a sequence of normalization decisions made *before* any model sees the data. Each step reduces variation — which helps generalization — but also discards information. The right combination depends entirely on the downstream task.

In this part you will implement individual preprocessing steps as composable functions, then assemble them into a configurable pipeline. The design goal is a pipeline where each step can be independently toggled — this is how preprocessing is done in production systems where the same codebase serves multiple tasks.

### 1.1 — Individual preprocessing functions

Write each function below, then write its docstring yourself (purpose, Args with types, Returns) — the stub only says `TODO: Write your own docstring`.

- `to_lowercase(text: str) -> str`
- `remove_punctuation(text: str) -> str` — use `string.punctuation` as the set of characters to strip.
- `tokenize_whitespace(text: str) -> list[str]` — split on whitespace only. Do not use `nltk.tokenize.word_tokenize` or any other NLTK tokenizer for this function.
- `remove_stopwords(tokens: list[str], stopword_set: set) -> list[str]`

In [3]:
_PUNCTUATION_TABLE = str.maketrans("", "", string.punctuation)


def _require_str(value, name):
    """Check that an argument is a string.

    Args:
        value (object): The argument to check.
        name (str): The argument's name, used in the error message.

    Returns:
        None

    Raises:
        TypeError: If ``value`` is not a ``str``.
    """
    if not isinstance(value, str):
        raise TypeError(f"{name} must be a str, got {type(value).__name__}.")


def _reject_bare_string(value, name):
    """Check that an argument meant to hold several strings is not one string.

    Iterating a string yields single characters, so raw text passed where a
    token sequence or corpus is expected would otherwise give silently wrong
    results.

    Args:
        value (Iterable[str]): The token sequence, corpus, or set to check.
        name (str): The argument's name, used in the error message.

    Returns:
        None

    Raises:
        TypeError: If ``value`` is a ``str``.
    """
    if isinstance(value, str):
        raise TypeError(
            f"{name} must be a collection of strings, not a single str; "
            "tokenize the text or wrap it in a list first."
        )


def to_lowercase(text):
    """Convert text to lowercase.

    Args:
        text (str): The input text.

    Returns:
        str: ``text`` with every cased character lowercased.

    Raises:
        TypeError: If ``text`` is not a string.
    """
    _require_str(text, "text")
    return text.lower()


def remove_punctuation(text):
    """Delete every character in ``string.punctuation`` from text.

    Args:
        text (str): The input text.

    Returns:
        str: ``text`` with all ASCII punctuation characters removed.

    Raises:
        TypeError: If ``text`` is not a string.
    """
    _require_str(text, "text")
    return text.translate(_PUNCTUATION_TABLE)


def tokenize_whitespace(text):
    """Split text into tokens on runs of whitespace.

    Args:
        text (str): The input text.

    Returns:
        list[str]: The whitespace-separated tokens; empty for blank text.

    Raises:
        TypeError: If ``text`` is not a string.
    """
    _require_str(text, "text")
    return text.split()


def remove_stopwords(tokens, stopword_set):
    """Drop every token that appears in a stopword set.

    Matching is exact and case-sensitive, so lowercase the tokens first
    when the stopword set is lowercase.

    Args:
        tokens (list[str]): The tokens to filter.
        stopword_set (set): The stopwords to remove.

    Returns:
        list[str]: The tokens not in ``stopword_set``, in their original
            order.

    Raises:
        TypeError: If ``tokens`` or ``stopword_set`` is a single string.
    """
    _reject_bare_string(tokens, "tokens")
    _reject_bare_string(stopword_set, "stopword_set")
    return [token for token in tokens if token not in stopword_set]


### 1.2 — Configurable preprocessing pipeline

Combine the functions above into a single pipeline function controlled by boolean flags. This design lets a caller select any combination of steps without modifying the function internals.

`preprocess(text: str, lowercase: bool = True, remove_punct: bool = True, remove_stops: bool = True, stopword_set: set | None = None) -> list[str]`

Call your four functions from 1.1 in this order — lowercase, remove punctuation, tokenize, remove stopwords — each only when its flag is True. If `stopword_set` is `None`, default it to `LAB_STOPWORDS`.

In [4]:
def preprocess(text, lowercase=True, remove_punct=True,
               remove_stops=True, stopword_set=None):
    """Run the configurable preprocessing pipeline on one document.

    Steps run in this order, each only when its flag is True: lowercase,
    remove punctuation, whitespace-tokenize (always), remove stopwords.

    Args:
        text (str): The raw document text.
        lowercase (bool): Lowercase the text. Defaults to True.
        remove_punct (bool): Strip ``string.punctuation`` characters.
            Defaults to True.
        remove_stops (bool): Remove stopwords after tokenizing.
            Defaults to True.
        stopword_set (set | None): Stopwords to remove. Defaults to None,
            which uses ``LAB_STOPWORDS``.

    Returns:
        list[str]: The document's tokens after the selected steps.

    Raises:
        TypeError: If ``text`` is not a string, or ``stopword_set`` is a
            single string.
    """
    if stopword_set is None:
        stopword_set = LAB_STOPWORDS
    if lowercase:
        text = to_lowercase(text)
    if remove_punct:
        text = remove_punctuation(text)
    tokens = tokenize_whitespace(text)
    if remove_stops:
        tokens = remove_stopwords(tokens, stopword_set)
    return tokens


# Quick test
tokens = preprocess(CORPUS_A[0])
print(f"Input:  '{CORPUS_A[0]}'")
print(f"Output: {tokens}")

Input:  'The dog barked at the mailman loudly'
Output: ['dog', 'barked', 'mailman', 'loudly']


### 1.3 — Compare pipeline variants

Apply four preprocessing configurations to every document in `CORPUS_A` and record the vocabulary size produced by each. Vocabulary size is a useful proxy for how aggressively each configuration collapses variation.

`get_vocab_size(corpus: list[str], **preprocess_kwargs) -> int` — tokenize every document in `corpus` using `preprocess(**preprocess_kwargs)` and return the number of unique token types across the whole corpus.

In [5]:
def get_vocab_size(corpus, **preprocess_kwargs):
    """Count the unique token types one preprocessing configuration produces.

    Args:
        corpus (list[str]): The documents to tokenize.
        **preprocess_kwargs (bool | set | None): Options forwarded to
            ``preprocess``: ``lowercase``, ``remove_punct``,
            ``remove_stops`` and ``stopword_set``.

    Returns:
        int: The number of distinct token types across the whole corpus.

    Raises:
        TypeError: If ``corpus`` is a single string or holds a non-string
            document.
    """
    _reject_bare_string(corpus, "corpus")
    return len({token for doc in corpus
                for token in preprocess(doc, **preprocess_kwargs)})


CONFIGS = {
    "baseline (no preprocessing)": dict(
        lowercase=False, remove_punct=False, remove_stops=False
    ),
    "lowercase only": dict(
        lowercase=True, remove_punct=False, remove_stops=False
    ),
    "lowercase + remove punctuation": dict(
        lowercase=True, remove_punct=True, remove_stops=False
    ),
    "full pipeline": dict(
        lowercase=True, remove_punct=True, remove_stops=True
    ),
}

vocab_sizes = {}
for name, kwargs in CONFIGS.items():
    vocab_sizes[name] = get_vocab_size(CORPUS_A, **kwargs)
    print(f"  {name:<40s}  vocab size: {vocab_sizes[name]}")

  baseline (no preprocessing)               vocab size: 27
  lowercase only                            vocab size: 26
  lowercase + remove punctuation            vocab size: 26
  full pipeline                             vocab size: 19


### 1.4 — Reflection (manually graded, 6 pts)

Answer in the markdown cell below. Your response should be 4–6 sentences.

**Prompt:** The four configurations produce different vocabulary sizes. For each of the following tasks, identify which configuration you would choose and explain *specifically* why that configuration helps and what you would lose by using the most aggressive one (full pipeline):
- **Task A:** Sentiment classification on product reviews
- **Task B:** Named entity recognition on news articles

Use precise terminology. "Removing stopwords reduces noise" earns 1 pt. "Removing stopwords collapses the token distribution which reduces sparsity for a bag-of-words sentiment classifier, but would cause an NER system to lose syntactic context words that signal entity boundaries" earns more.

**Your reflection:**

CORPUS_A has 32 tokens, and the four configurations reduce them to 27, 26, 26 and 19 word types: lowercasing only merges *The* with *the*, punctuation removal changes nothing because the corpus has no punctuation, and stopword removal deletes seven function-word types. For sentiment classification of product reviews I would choose lowercase + remove punctuation, because folding case and stripping the punctuation that whitespace splitting leaves attached (so *Great!* and *great* become one type) reduces sparsity in a bag-of-words classifier, while keeping stopwords protects negation words like *not* and *no* that flip the polarity of the words after them (J&M, *Naive Bayes Classification*, report that stop lists rarely improve text classification and that negation needs explicit handling). The price of this configuration is lost intensity: *awful!!!* collapses to *awful*, an emoticon such as `:(` disappears entirely because both characters are in `string.punctuation`, and lowercasing merges a shouted *AWFUL* with plain *awful*, which is acceptable for positive/negative polarity, where the sentiment words carry most of the signal, but not for predicting a 1–5 star rating, where I would keep *!* and emoticons as their own tokens. The full pipeline is risky for sentiment because standard stop lists, NLTK's English list among them, include *not*, *no*, *don't* and *but*; LAB_STOPWORDS happens to keep *not* and *no*, but it still drops *but*, which carries the contrast in "great screen but awful battery." For NER on news articles I would choose the baseline, because the full pipeline throws away capitalization, the word-shape cue that separates *Apple* the company from *apple* the fruit, and deletes function words that sit inside names, such as *of* in *Bank of America*, or that signal an entity boundary, such as *in* before a location (J&M, *Sequence Labeling for Parts of Speech and Named Entities*). Removing punctuation would also corrupt entities, turning *U.S.* into *us* once lowercased and *AT&T* into *att*, so for NER I would split punctuation off as separate tokens rather than delete it.

---

## Part 2 — Tokenization Strategies (20 points)

Word-level tokenization is intuitive but fragile: it cannot handle unknown words, and it treats every surface form as a distinct type. Subword tokenization — used by virtually every modern language model — splits rare or unknown words into smaller units that the model has seen during training. Two dominant algorithms are **Byte-Pair Encoding (BPE)** and **WordPiece**.

In this part you will implement a simplified BPE learner from scratch and use the Hugging Face `tokenizers` library to run WordPiece, then compare their outputs and explain when each is appropriate.

### 2.1 — Word-level tokenization (baseline)

`word_tokenize_corpus(corpus: list[str], lowercase: bool = True) -> list[list[str]]` — call `preprocess()` from Part 1 with `remove_punct=False` and `remove_stops=False` (we want every token here, not a reduced vocabulary).

In [6]:
def word_tokenize_corpus(corpus, lowercase=True):
    """Split each document into word tokens, keeping punctuation and stopwords.

    Args:
        corpus (list[str]): The documents to tokenize.
        lowercase (bool): Lowercase the text before splitting.
            Defaults to True.

    Returns:
        list[list[str]]: One list of whitespace-separated tokens per
            document.

    Raises:
        TypeError: If ``corpus`` is a single string or holds a non-string
            document.
    """
    _reject_bare_string(corpus, "corpus")
    return [preprocess(doc, lowercase=lowercase, remove_punct=False,
                       remove_stops=False)
            for doc in corpus]


word_tokens_A = word_tokenize_corpus(CORPUS_A)
word_vocab_A = sorted(set(t for doc in word_tokens_A for t in doc))

print("Word-level tokenization of Corpus A:")
print(f"  Total tokens: {sum(len(d) for d in word_tokens_A)}")
print(f"  Unique types: {len(word_vocab_A)}")
print(f"  Doc 0 tokens: {word_tokens_A[0]}")

Word-level tokenization of Corpus A:
  Total tokens: 32
  Unique types: 26
  Doc 0 tokens: ['the', 'dog', 'barked', 'at', 'the', 'mailman', 'loudly']


### 2.2 — Byte-Pair Encoding (BPE) from scratch

BPE starts by treating every word as a sequence of characters (with a special end-of-word marker `</w>`), then iteratively merges the most frequent adjacent pair until a target vocabulary size is reached.

Implement the functions below. `build_bpe_vocab`, `get_bpe_pairs`, and `merge_bpe_pair` are the three core operations; `learn_bpe` orchestrates them into a training loop.

- `build_bpe_vocab(corpus: list[str]) -> Counter` — maps a character-segmented word string to its corpus frequency. Represent each word as its characters space-separated with `</w>` appended to mark the word boundary, e.g. `'dog'` -> `'d o g </w>'`.
- `get_bpe_pairs(vocab: Counter) -> Counter` — maps each adjacent symbol pair (as a 2-tuple) to its total frequency across all words in `vocab`.
- `merge_bpe_pair(pair: tuple[str, str], vocab: Counter) -> Counter` — returns a new vocabulary with every occurrence of `pair` merged into a single symbol.
- `learn_bpe(corpus: list[str], num_merges: int) -> tuple[list[tuple], Counter]` — build the initial vocabulary with `build_bpe_vocab`, then repeat `num_merges` times: find the most frequent pair with `get_bpe_pairs` and merge it with `merge_bpe_pair`, recording each merge rule in order. Returns `(merges, final_vocab)`.

In [7]:
END_OF_WORD = "</w>"


def _word_to_symbols(word):
    """Split a word into characters and append the end-of-word marker.

    Args:
        word (str): The word to segment.

    Returns:
        list[str]: The word's characters followed by ``END_OF_WORD``,
            e.g. ``'dog'`` -> ``['d', 'o', 'g', '</w>']``.

    Raises:
        TypeError: If ``word`` is not a string.
    """
    _require_str(word, "word")
    return list(word) + [END_OF_WORD]


def _merge_symbol_pair(symbols, pair):
    """Join every adjacent occurrence of a symbol pair in one segmentation.

    Args:
        symbols (list[str]): The current symbols of one word.
        pair (tuple[str, str]): The two adjacent symbols to join.

    Returns:
        list[str]: A new symbol list in which each occurrence of ``pair``,
            scanned left to right, is one symbol.
    """
    first, second = pair
    merged = []
    i = 0
    while i < len(symbols):
        # Match whole symbols, never substrings: str.replace('a b', 'ab')
        # would also fuse the end of a longer symbol, turning 'ca b' into
        # 'cab'.
        if (i + 1 < len(symbols)
                and symbols[i] == first and symbols[i + 1] == second):
            merged.append(first + second)
            i += 2
        else:
            merged.append(symbols[i])
            i += 1
    return merged


def build_bpe_vocab(corpus):
    """Build the initial BPE vocabulary of character-segmented words.

    Args:
        corpus (list[str]): The training documents.

    Returns:
        Counter: Maps each word, written as space-separated characters plus
            ``'</w>'`` (e.g. ``'d o g </w>'``), to its corpus frequency.

    Raises:
        TypeError: If ``corpus`` is a single string or holds a non-string
            document.
    """
    # The spec leaves word splitting and casing open. Reusing the 2.1
    # tokenizer (lowercase, whitespace split) trains BPE on the same words
    # as the word-level baseline and matches the lowercase test words in 2.3.
    word_counts = Counter(word for doc in word_tokenize_corpus(corpus)
                          for word in doc)
    return Counter({" ".join(_word_to_symbols(word)): count
                    for word, count in word_counts.items()})


def get_bpe_pairs(vocab):
    """Count adjacent symbol pairs, weighted by word frequency.

    Args:
        vocab (Counter): Maps space-separated symbol strings to word
            frequencies, as returned by ``build_bpe_vocab``.

    Returns:
        Counter: Maps each adjacent ``(left, right)`` symbol pair to the
            summed frequency of the words it occurs in, counted once per
            occurrence.
    """
    pair_counts = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for pair in zip(symbols, symbols[1:]):
            pair_counts[pair] += freq
    return pair_counts


def merge_bpe_pair(pair, vocab):
    """Merge one symbol pair everywhere it occurs in a BPE vocabulary.

    Args:
        pair (tuple[str, str]): The adjacent symbols to merge.
        vocab (Counter): Maps space-separated symbol strings to word
            frequencies.

    Returns:
        Counter: A new vocabulary in which each occurrence of ``pair`` is
            one symbol. ``vocab`` is left unchanged.
    """
    merged_vocab = Counter()
    for word, freq in vocab.items():
        merged_vocab[" ".join(_merge_symbol_pair(word.split(), pair))] += freq
    return merged_vocab


def learn_bpe(corpus, num_merges):
    """Learn an ordered list of BPE merge rules from a corpus.

    Args:
        corpus (list[str]): The training documents.
        num_merges (int): The maximum number of merges to learn. Training
            stops early once no adjacent pairs remain.

    Returns:
        tuple[list[tuple[str, str]], Counter]: ``(merges, final_vocab)``:
            the merge rules in the order they were learned, and the
            vocabulary after all of them were applied.

    Raises:
        ValueError: If ``num_merges`` is negative.
        TypeError: If ``corpus`` is a single string or holds a non-string
            document.
    """
    if num_merges < 0:
        raise ValueError(f"num_merges must be non-negative, got {num_merges}.")
    vocab = build_bpe_vocab(corpus)
    merges = []
    for _ in range(num_merges):
        pair_counts = get_bpe_pairs(vocab)
        if not pair_counts:
            break
        # Among tied pairs, most_common(1) returns the first one encountered.
        best_pair = pair_counts.most_common(1)[0][0]
        vocab = merge_bpe_pair(best_pair, vocab)
        merges.append(best_pair)
    return merges, vocab


NUM_MERGES = 10
bpe_merges_A, bpe_vocab_A = learn_bpe(CORPUS_A, NUM_MERGES)

print(f"BPE: learned {len(bpe_merges_A)} merges on Corpus A")
print("Merge rules (in order):")
for i, merge in enumerate(bpe_merges_A, 1):
    print(f"  {i:2d}: {merge[0]!r} + {merge[1]!r} -> {merge[0]+merge[1]!r}")

BPE: learned 10 merges on Corpus A
Merge rules (in order):
   1: 'e' + '</w>' -> 'e</w>'
   2: 's' + '</w>' -> 's</w>'
   3: 't' + 'h' -> 'th'
   4: 'th' + 'e</w>' -> 'the</w>'
   5: 'a' + 't' -> 'at'
   6: 'g' + '</w>' -> 'g</w>'
   7: 'a' + 'r' -> 'ar'
   8: 'at' + '</w>' -> 'at</w>'
   9: 'm' + 'a' -> 'ma'
  10: 'n' + '</w>' -> 'n</w>'


### 2.3 — Apply BPE to segment a new word

Use the learned merge rules to segment a word not seen during training. This demonstrates how BPE handles out-of-vocabulary words — it falls back to character-level segments and applies merges in order.

`apply_bpe(word: str, merges: list[tuple]) -> list[str]` — start from the word's character-level segmentation (with the same `</w>` end marker used in training), then apply each rule in `merges`, in order, merging that pair wherever it currently appears.

In [8]:
def apply_bpe(word, merges):
    """Segment a word by replaying learned BPE merges in order.

    An unseen word keeps whatever character-level pieces the merges cannot
    join. The merges were learned on lowercased text, so pass a lowercased
    word.

    Args:
        word (str): The word to segment.
        merges (list[tuple[str, str]]): Merge rules in the order learned.

    Returns:
        list[str]: The word's subword symbols. The last one carries the
            end-of-word marker, either alone (``'</w>'``) or fused into a
            merged symbol (e.g. ``'ed</w>'``).

    Raises:
        TypeError: If ``word`` is not a string.
    """
    symbols = _word_to_symbols(word)
    for pair in merges:
        symbols = _merge_symbol_pair(symbols, pair)
    return symbols


# Test on words from Corpus A and one unseen word
test_words = ['barked', 'delivering', 'mailman']
for w in test_words:
    seg = apply_bpe(w, bpe_merges_A)
    print(f"  {w:15s} -> {seg}")

  barked          -> ['b', 'ar', 'k', 'e', 'd', '</w>']
  delivering      -> ['d', 'e', 'l', 'i', 'v', 'e', 'r', 'i', 'n', 'g</w>']
  mailman         -> ['ma', 'i', 'l', 'ma', 'n</w>']


### 2.4 — WordPiece via Hugging Face tokenizers

WordPiece (used by BERT) differs from BPE in its merge criterion: instead of most frequent pair, it selects the pair that maximizes the likelihood of the training data under a unigram language model. In practice this tends to produce more linguistically meaningful subwords.

We use the `tokenizers` library to train a WordPiece tokenizer on `CORPUS_A` and examine its output. **This is a `[RUN]` cell — do not modify it.**

In [9]:
# [RUN] — WordPiece via Hugging Face tokenizers library.
# Do not modify. Examine the output and answer the reflection below.

from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

wp_tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
wp_tokenizer.pre_tokenizer = Whitespace()

trainer = WordPieceTrainer(
    vocab_size=60,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
)
wp_tokenizer.train_from_iterator(CORPUS_A, trainer=trainer)

print("WordPiece tokenization of Corpus A documents:")
for i, doc in enumerate(CORPUS_A):
    encoding = wp_tokenizer.encode(doc.lower())
    print(f"  Doc {i}: {encoding.tokens}")

# Vocabulary size
wp_vocab = wp_tokenizer.get_vocab()
wp_vocab_size_A = len(wp_vocab)
print(f"\nWordPiece vocab size: {wp_vocab_size_A}")

# Test on an unseen word (same as BPE test)
test_word = 'delivering'
wp_seg = wp_tokenizer.encode(test_word).tokens
print(f"\nWordPiece segmentation of '{test_word}': {wp_seg}")




WordPiece tokenization of Corpus A documents:
  Doc 0: ['the', 'dog', 'bar', '##k', '##e', '##d', 'a', '##t', 'the', 'ma', '##i', '##l', '##m', '##a', '##n', 'l', '##ou', '##d', '##ly']
  Doc 1: ['the', 'cat', 's', '##at', 'o', '##n', 'the', 'ma', '##t', 'q', '##u', '##i', '##et', '##ly']
  Doc 2: ['dog', '##s', 'a', '##n', '##d', 'cat', '##s', 'a', '##r', '##e', 'c', '##o', '##m', '##m', '##o', '##n', 'h', '##ou', '##s', '##e', '##h', '##o', '##l', '##d', 'p', '##et', '##s']
  Doc 3: ['the', 'ma', '##i', '##l', '##m', '##a', '##n', 'd', '##e', '##l', '##i', '##v', '##er', '##s', 'l', '##et', '##t', '##er', '##s', 'e', '##v', '##er', '##y', 'm', '##o', '##r', '##n', '##i', '##n', '##g']
  Doc 4: ['a', 'bar', '##k', '##i', '##n', '##g', 'dog', 'r', '##ar', '##e', '##ly', 'b', '##i', '##t', '##e', '##s']

WordPiece vocab size: 60

WordPiece segmentation of 'delivering': ['d', '##e', '##l', '##i', '##v', '##er', '##i', '##n', '##g']


### 2.5 — Reflection (manually graded, 6 pts)

Answer in the markdown cell below. 4–6 sentences total.

**Prompt:** Compare BPE and WordPiece on the outputs you observed above. Then answer: for each of the following scenarios, which tokenization strategy — word-level, BPE, or WordPiece — would you choose and why?
- A language model being pre-trained on a large English news corpus
- A domain-specific medical NER system where morphological suffixes (e.g. *-itis*, *-ectomy*) carry semantic meaning
- A multilingual model covering languages with rich morphology (Turkish, Finnish)

Your answer should go beyond "subword is better for OOV" — explain the mechanism.

**Your reflection:**

My ten BPE merges are driven purely by pair frequency and mostly attach the end-of-word marker to common endings (`e</w>`, `s</w>`, `g</w>`, `n</w>`) or build short pieces (`th`, then `the</w>`, `at`, `ar`, `ma`), so 'delivering' comes out as nine single characters plus `g</w>`; WordPiece instead marks word-internal continuation pieces with `##`, learned word-initial units such as `the`, `dog`, `cat` and `bar`, and splits 'delivering' as `d ##e ##l ##i ##v ##er ##i ##n ##g`. The outputs look alike because, although WordPiece as an algorithm picks the merge that most increases the likelihood of the training data, the installed `tokenizers` 0.23.2 source (`models/wordpiece/trainer.rs`) shows that `WordPieceTrainer` simply runs the frequency-count `BpeTrainer` and converts its result with `WordPiece::from_bpe`, so what differs is the `##` versus `</w>` notation and WordPiece's greedy longest-match encoding (even its last merge is a frequency tie that changed between my runs; this run learned `##ou`, an earlier run `##an`). For pre-training a language model on a large English news corpus I would use byte-level BPE, because frequent words become single tokens while any rare name or number still breaks into known pieces, so there are no unknown words and the number of merges directly trades vocabulary size against sequence length (J&M, *Words and Tokens*). For medical NER I would train a WordPiece vocabulary on clinical text and tag with a BERT-style encoder: because merges follow frequency, only an in-domain corpus makes *itis* and *ectomy* frequent enough to become pieces (general-English BERT splits 'appendectomy' into `app ##end ##ect ##omy`), and the first-piece vector that labels each word can attend to a `##ectomy` piece that signals a procedure (J&M, *Masked Language Models*). For Turkish and Finnish, word-level tokenization fails because agglutinative morphology stacks affixes onto a root (*ev-ler-iniz-den*, 'from your houses'), so most word types are unseen at test time, and I would use BPE trained on a language-balanced sample so the shared merges learn each language's frequent affixes as reusable pieces rather than spending the vocabulary on English and fragmenting Turkish words into characters.

---

## Part 3 — Normalization: Stemming and Lemmatization (18 points)

Both stemming and lemmatization reduce tokens to a base form, collapsing morphological variation. Stemming is a rule-based heuristic that chops affixes; it is fast but produces non-words. Lemmatization uses a dictionary lookup to return the canonical dictionary form; it is slower but linguistically accurate.

### 3.1 — Apply stemming and lemmatization

- `stem_tokens(tokens: list[str]) -> list[str]` — apply Porter stemming (the `stemmer` object above) to every token.
- `lemmatize_tokens(tokens: list[str], pos: str = 'v') -> list[str]` — apply WordNet lemmatization (the `lemmatizer` object above) to every token, using `pos` as the part-of-speech hint (`'v'` verb, `'n'` noun, `'a'` adjective).

In [10]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

WORDNET_POS_TAGS = frozenset({"n", "v", "a", "r", "s"})


def stem_tokens(tokens):
    """Reduce each token to its Porter stem.

    Args:
        tokens (list[str]): The tokens to stem.

    Returns:
        list[str]: The Porter stem of each token, in order. Stems need not
            be dictionary words.

    Raises:
        TypeError: If ``tokens`` is a single string.
    """
    _reject_bare_string(tokens, "tokens")
    return [stemmer.stem(token) for token in tokens]


def lemmatize_tokens(tokens, pos='v'):
    """Map each token to its WordNet lemma under one part of speech.

    Args:
        tokens (list[str]): The tokens to lemmatize.
        pos (str): WordNet part-of-speech tag applied to every token:
            ``'n'`` noun, ``'v'`` verb, ``'a'`` adjective, ``'r'`` adverb,
            or ``'s'`` satellite adjective. Defaults to ``'v'``.

    Returns:
        list[str]: The lemma of each token, in order. A token WordNet does
            not know under ``pos`` is returned unchanged.

    Raises:
        TypeError: If ``tokens`` is a single string.
        ValueError: If ``pos`` is not a WordNet part-of-speech tag.
    """
    _reject_bare_string(tokens, "tokens")
    if pos not in WORDNET_POS_TAGS:
        raise ValueError(
            f"pos must be one of {sorted(WORDNET_POS_TAGS)}, got {pos!r}."
        )
    return [lemmatizer.lemmatize(token, pos=pos) for token in tokens]


# Test on controlled word list
TEST_WORDS = [
    'running', 'dogs', 'barked', 'quietly', 'delivering', 'cats',
    'bites',
]

stemmed_test = stem_tokens(TEST_WORDS)
lemmatized_test = lemmatize_tokens(TEST_WORDS)

print(f"{'Word':15s} {'Stemmed':15s} {'Lemmatized':15s}")
print("-" * 45)
for word, stemmed, lemma in zip(TEST_WORDS, stemmed_test, lemmatized_test):
    print(f"{word:15s} {stemmed:15s} {lemma:15s}")

Word            Stemmed         Lemmatized     
---------------------------------------------
running         run             run            
dogs            dog             dog            
barked          bark            bark           
quietly         quietli         quietly        
delivering      deliv           deliver        
cats            cat             cat            
bites           bite            bite           


### 3.2 — Vocabulary size impact

Apply four normalization configurations to `CORPUS_A` (after full preprocessing) and compare the resulting vocabulary sizes.

`preprocess_and_normalize(corpus: list[str], use_stemming: bool = False, use_lemmatization: bool = False) -> list[list[str]]` — run the full preprocessing pipeline (lowercase, remove punctuation, remove stopwords) on every document, then apply stemming if `use_stemming` is True, or lemmatization if `use_lemmatization` is True. Raise a `ValueError` if both flags are True — the two are mutually exclusive.

In [11]:
def preprocess_and_normalize(corpus, use_stemming=False,
                              use_lemmatization=False):
    """Fully preprocess each document, then optionally stem or lemmatize it.

    Args:
        corpus (list[str]): The documents to process.
        use_stemming (bool): Apply Porter stemming. Defaults to False.
        use_lemmatization (bool): Apply WordNet lemmatization with the
            default verb tag. Defaults to False.

    Returns:
        list[list[str]]: One list of normalized tokens per document.

    Raises:
        ValueError: If both ``use_stemming`` and ``use_lemmatization`` are
            True.
        TypeError: If ``corpus`` is a single string or holds a non-string
            document.
    """
    if use_stemming and use_lemmatization:
        raise ValueError(
            "use_stemming and use_lemmatization are mutually exclusive; "
            "set at most one of them to True."
        )
    _reject_bare_string(corpus, "corpus")
    tokenized_docs = [preprocess(doc) for doc in corpus]
    if use_stemming:
        return [stem_tokens(doc) for doc in tokenized_docs]
    if use_lemmatization:
        return [lemmatize_tokens(doc) for doc in tokenized_docs]
    return tokenized_docs


# Named output variables for the autograder
tokens_no_norm = preprocess_and_normalize(CORPUS_A, False, False)
tokens_stemmed = preprocess_and_normalize(CORPUS_A, True, False)
tokens_lemmatized = preprocess_and_normalize(CORPUS_A, False, True)

vocab_no_norm = sorted(set(t for doc in tokens_no_norm for t in doc))
vocab_stemmed = sorted(set(t for doc in tokens_stemmed for t in doc))
vocab_lemmatized = sorted(
    set(t for doc in tokens_lemmatized for t in doc)
)

vocab_size_no_norm = len(vocab_no_norm)
vocab_size_stemmed = len(vocab_stemmed)
vocab_size_lemmatized = len(vocab_lemmatized)

print(f"Vocab size — no normalization:  {vocab_size_no_norm}")
print(f"Vocab size — stemmed:           {vocab_size_stemmed}")
print(f"Vocab size — lemmatized:        {vocab_size_lemmatized}")

Vocab size — no normalization:  19
Vocab size — stemmed:           16
Vocab size — lemmatized:        16


### 3.3 — Reflection (manually graded, 6 pts)

**Prompt:** Stemming reduced the vocabulary more aggressively than lemmatization. In the markdown cell below, explain:
- One task where stemming's more aggressive reduction is beneficial and the loss of valid word forms is acceptable
- One task where lemmatization is strongly preferred over stemming, and explain the specific failure mode that stemming would introduce
- Whether either technique is appropriate for a transformer-based model that uses WordPiece tokenization, and why

4–6 sentences. Use specific examples from the word list printed above.

**Your reflection:**

On CORPUS_A stemming was not actually more aggressive than lemmatization: both cut the vocabulary from 19 to 16 types by merging *barked/barking*, *dog/dogs* and *cat/cats*, so the difference shows in the printed table instead, where Porter leaves the derivational suffix *-ly* of *quietly* in place and only rewrites its final *y* as *i*, giving the non-word *quietli*, and cuts *delivering* to *deliv*, while the WordNet lemmatizer returns the real lemmas *quietly* and *deliver*. Stemming's aggressiveness is worth it for a bag-of-words search engine, where recall matters and no user ever sees a stem: a query for "delivering" should match documents containing *delivers*, and mapping both to *deliv* also reduces sparsity in the term-document matrix. Lemmatization is strongly preferred when normalized tokens must be dictionary words, for instance as keys into a sentiment lexicon or into WordNet for word-sense lookup, because *quietli* and *deliv* are unknown words to any lexicon, and Porter's suffix rules both over-stem unrelated words together (*university* and *universe* both become *univers*) and under-stem related ones apart (*delivery* becomes *deliveri*, not *deliv*). For a transformer with WordPiece tokenization I would apply neither, because its subword vocabulary and pretrained embeddings were learned on raw text: BERT keeps *quietly* and *delivering* as single tokens but splits *quietli* into `quiet ##li` and *deliv* into `del ##iv`, so the stems become unfamiliar input. Stemming would also delete inflectional morphemes such as the past-tense *-ed* in *barked* and the plural *-s* in *dogs* that the model uses for tense and agreement, all to fix a sparsity problem the model does not have, since WordPiece already breaks rare inflected forms into known pieces (J&M, *Words and Tokens*; *Embeddings*).

---

## Part 4 — Bag of Words and TF-IDF from Scratch (27 points)

Bag of Words and TF-IDF are the two foundational text representations in NLP. BoW counts how often each vocabulary term appears in a document; TF-IDF reweights those counts to discount terms that are common across many documents and reward terms that are distinctive.

You will implement both representations using only NumPy and Python built-ins. After verifying your implementation is correct, you will compare it against `sklearn`'s output to understand exactly how the library's smoothed variant differs.

**Important:** Do not use `sklearn.CountVectorizer`, `sklearn.TfidfVectorizer`, or any other library that computes these representations for you.

### 4.1 — Build vocabulary and BoW matrix

- `build_vocabulary(tokenized_docs: list[list[str]]) -> dict[str, int]` — maps each unique token across all documents to a 0-indexed column position. Sort tokens alphabetically first so the mapping is deterministic.
- `build_bow_matrix(tokenized_docs: list[list[str]], vocab: dict[str, int]) -> np.ndarray` — an integer matrix of shape `(n_docs, vocab_size)` where entry `[i, j]` is how many times vocabulary term `j` appears in document `i`.

In [12]:
def _as_token_lists(tokenized_docs):
    """Copy tokenized documents into a list of token lists.

    Args:
        tokenized_docs (list[list[str]]): One token sequence per document.

    Returns:
        list[list[str]]: Each document's tokens as a list.

    Raises:
        TypeError: If ``tokenized_docs`` or any document in it is a single
            string, which means the text was never tokenized.
    """
    _reject_bare_string(tokenized_docs, "tokenized_docs")
    token_lists = []
    for index, doc in enumerate(tokenized_docs):
        _reject_bare_string(doc, f"tokenized_docs[{index}]")
        token_lists.append(list(doc))
    return token_lists


def build_vocabulary(tokenized_docs):
    """Assign each unique token a column index in alphabetical order.

    Args:
        tokenized_docs (list[list[str]]): One token list per document.

    Returns:
        dict[str, int]: Maps each unique token to its 0-based column, with
            columns in sorted token order.

    Raises:
        TypeError: If ``tokenized_docs`` or any document in it is a single
            string.
    """
    token_lists = _as_token_lists(tokenized_docs)
    unique_tokens = sorted({token for doc in token_lists for token in doc})
    return {token: index for index, token in enumerate(unique_tokens)}


def build_bow_matrix(tokenized_docs, vocab):
    """Count how often each vocabulary term occurs in each document.

    Tokens missing from ``vocab`` are out-of-vocabulary and are skipped, so
    a vocabulary built on a different corpus still yields a valid matrix.

    Args:
        tokenized_docs (list[list[str]]): One token list per document.
        vocab (dict[str, int]): Maps each term to its column index, as
            returned by ``build_vocabulary``.

    Returns:
        np.ndarray: Integer count matrix of shape ``(n_docs, len(vocab))``
            whose entry ``[i, j]`` is how often term ``j`` occurs in
            document ``i``.

    Raises:
        TypeError: If ``tokenized_docs`` or any document in it is a single
            string.
    """
    token_lists = _as_token_lists(tokenized_docs)
    bow = np.zeros((len(token_lists), len(vocab)), dtype=np.int64)
    for row, doc in enumerate(token_lists):
        for token, count in Counter(doc).items():
            if token in vocab:
                bow[row, vocab[token]] = count
    return bow


# Apply to fully preprocessed Corpus A (using tokens_no_norm from Part 3)
vocab_A = build_vocabulary(tokens_no_norm)
bow_A = build_bow_matrix(tokens_no_norm, vocab_A)

print(f"Vocabulary size: {len(vocab_A)}")
print(f"BoW matrix shape: {bow_A.shape}  (docs × vocab)")
print(f"\nFirst 8 vocab terms: {list(vocab_A.keys())[:8]}")
print(f"Doc 0 counts:       {bow_A[0, :8]}")
print(
    f"Doc 0 non-zero terms: "
    f"{[w for w, i in vocab_A.items() if bow_A[0, i] > 0]}"
)

Vocabulary size: 19
BoW matrix shape: (5, 19)  (docs × vocab)

First 8 vocab terms: ['barked', 'barking', 'bites', 'cat', 'cats', 'common', 'delivers', 'dog']
Doc 0 counts:       [1 0 0 0 0 0 0 1]
Doc 0 non-zero terms: ['barked', 'dog', 'loudly', 'mailman']


### 4.2 — Term frequency (TF)

Term frequency normalizes raw counts by document length, so longer documents do not automatically get higher scores.

`compute_tf(bow_matrix: np.ndarray) -> np.ndarray` — TF(t, d) = count(t, d) / total_tokens(d). Returns a float matrix the same shape as `bow_matrix`, where each row sums to 1.0. Guard against division by zero for a document with zero tokens — its row should be all zeros, not NaN.

In [13]:
def _as_2d_float(matrix, name):
    """Convert an array-like to a 2-D float array.

    Args:
        matrix (np.ndarray): The matrix to convert, one row per document.
            Nested lists are accepted too.
        name (str): The argument's name, used in the error message.

    Returns:
        np.ndarray: ``matrix`` as a 2-D float array.

    Raises:
        ValueError: If ``matrix`` is not 2-D.
    """
    array = np.asarray(matrix, dtype=float)
    if array.ndim != 2:
        raise ValueError(
            f"{name} must be 2-D (n_docs, vocab_size), got shape "
            f"{array.shape}."
        )
    return array


def compute_tf(bow_matrix):
    """Normalize raw term counts by document length.

    Args:
        bow_matrix (np.ndarray): Term counts of shape
            ``(n_docs, vocab_size)``.

    Returns:
        np.ndarray: Float matrix of the same shape with
            ``TF[i, j] = count[i, j] / total_tokens[i]``. Each row sums to
            1.0, except that a document with no tokens gets a row of zeros.

    Raises:
        ValueError: If ``bow_matrix`` is not 2-D.
    """
    counts = _as_2d_float(bow_matrix, "bow_matrix")
    doc_lengths = counts.sum(axis=1, keepdims=True)
    # An empty document has length 0. Skipping it avoids 0 / 0 = NaN and
    # leaves its row at the zeros preset in `out`.
    return np.divide(counts, doc_lengths, out=np.zeros_like(counts),
                     where=doc_lengths > 0)


tf_A = compute_tf(bow_A)

print(f"TF matrix shape: {tf_A.shape}")
print(f"TF doc 0 (first 8 terms): {np.round(tf_A[0, :8], 4)}")
print(f"TF doc 0 row sum: {tf_A[0].sum():.6f}  (should be 1.0)")

TF matrix shape: (5, 19)
TF doc 0 (first 8 terms): [0.25 0.   0.   0.   0.   0.   0.   0.25]
TF doc 0 row sum: 1.000000  (should be 1.0)


### 4.3 — Inverse document frequency (IDF)

IDF down-weights terms that appear in many documents and up-weights terms that are distinctive.

**Use the classic formula:**

$$\text{IDF}(t) = \log\left(\frac{N}{df(t)}\right)$$

where $N$ is the number of documents and $df(t)$ is the number of documents containing term $t$. Use the **natural logarithm**. Terms appearing in every document will have IDF = 0.

`compute_idf(bow_matrix: np.ndarray) -> np.ndarray` — returns a 1-D float array of length `vocab_size`.

In [14]:
def compute_idf(bow_matrix):
    """Compute the classic inverse document frequency of every term.

    ``IDF(t) = ln(N / df(t))``, where ``N`` is the number of documents and
    ``df(t)`` is the number of documents containing ``t``. A term that
    occurs in every document gets IDF 0.

    Args:
        bow_matrix (np.ndarray): Term counts of shape
            ``(n_docs, vocab_size)``.

    Returns:
        np.ndarray: 1-D float array of length ``vocab_size``. A term with
            ``df = 0`` gets 0.0 instead of ``inf``; that only happens when
            the vocabulary was built from a different corpus than the
            matrix.

    Raises:
        ValueError: If ``bow_matrix`` is not 2-D.
    """
    counts = _as_2d_float(bow_matrix, "bow_matrix")
    n_docs = counts.shape[0]
    doc_freq = (counts > 0).sum(axis=0)
    idf = np.zeros(counts.shape[1])
    # ln(N / 0) is inf, so terms that occur in no document stay at 0.0.
    seen = doc_freq > 0
    idf[seen] = np.log(n_docs / doc_freq[seen])
    return idf


idf_A = compute_idf(bow_A)

print(f"IDF vector length: {len(idf_A)}")
print(f"IDF values (first 8 terms): {np.round(idf_A[:8], 4)}")

# Show a few informative examples
terms = list(vocab_A.keys())
for term, idf_val in sorted(zip(terms, idf_A), key=lambda x: x[1]):
    df_val = int((bow_A[:, vocab_A[term]] > 0).sum())
    print(f"  IDF('{term}') = log({len(bow_A)}/{df_val}) = {idf_val:.4f}")

IDF vector length: 19
IDF values (first 8 terms): [1.6094 1.6094 1.6094 1.6094 1.6094 1.6094 1.6094 0.9163]
  IDF('dog') = log(5/2) = 0.9163
  IDF('mailman') = log(5/2) = 0.9163
  IDF('barked') = log(5/1) = 1.6094
  IDF('barking') = log(5/1) = 1.6094
  IDF('bites') = log(5/1) = 1.6094
  IDF('cat') = log(5/1) = 1.6094
  IDF('cats') = log(5/1) = 1.6094
  IDF('common') = log(5/1) = 1.6094
  IDF('delivers') = log(5/1) = 1.6094
  IDF('dogs') = log(5/1) = 1.6094
  IDF('household') = log(5/1) = 1.6094
  IDF('letters') = log(5/1) = 1.6094
  IDF('loudly') = log(5/1) = 1.6094
  IDF('mat') = log(5/1) = 1.6094
  IDF('morning') = log(5/1) = 1.6094
  IDF('pets') = log(5/1) = 1.6094
  IDF('quietly') = log(5/1) = 1.6094
  IDF('rarely') = log(5/1) = 1.6094
  IDF('sat') = log(5/1) = 1.6094


### 4.4 — TF-IDF matrix

`compute_tfidf(tf_matrix: np.ndarray, idf_vector: np.ndarray) -> np.ndarray` — TF-IDF(t, d) = TF(t, d) * IDF(t). Multiply each row of `tf_matrix` by `idf_vector` element-wise. Returns a float matrix the same shape as `tf_matrix`.

In [15]:
def compute_tfidf(tf_matrix, idf_vector):
    """Weight each term frequency by its term's inverse document frequency.

    Args:
        tf_matrix (np.ndarray): Term frequencies of shape
            ``(n_docs, vocab_size)``.
        idf_vector (np.ndarray): IDF values of shape ``(vocab_size,)``.

    Returns:
        np.ndarray: Float matrix of the same shape as ``tf_matrix`` with
            ``TFIDF[i, j] = TF[i, j] * IDF[j]``.

    Raises:
        ValueError: If ``tf_matrix`` is not 2-D, or ``idf_vector`` is not
            1-D with one entry per column of ``tf_matrix``.
    """
    tf = _as_2d_float(tf_matrix, "tf_matrix")
    idf = np.asarray(idf_vector, dtype=float)
    if idf.shape != (tf.shape[1],):
        raise ValueError(
            f"idf_vector must have shape ({tf.shape[1]},) to match the "
            f"columns of tf_matrix, got {idf.shape}."
        )
    # The (vocab_size,) vector broadcasts across every row, so column j of
    # each document is scaled by idf[j].
    return tf * idf


tfidf_A = compute_tfidf(tf_A, idf_A)

print(f"TF-IDF matrix shape: {tfidf_A.shape}")
print(f"TF-IDF doc 0 (first 8): {np.round(tfidf_A[0, :8], 4)}")

# Top-scored terms per document
print("\nTop TF-IDF term per document:")
for i, doc in enumerate(CORPUS_A):
    top_idx = np.argmax(tfidf_A[i])
    top_term = terms[top_idx]
    print(f"  Doc {i}: '{top_term}' ({tfidf_A[i, top_idx]:.4f})  |  {doc}")

TF-IDF matrix shape: (5, 19)
TF-IDF doc 0 (first 8): [0.4024 0.     0.     0.     0.     0.     0.     0.2291]

Top TF-IDF term per document:
  Doc 0: 'barked' (0.4024)  |  The dog barked at the mailman loudly
  Doc 1: 'cat' (0.4024)  |  The cat sat on the mat quietly
  Doc 2: 'cats' (0.3219)  |  Dogs and cats are common household pets
  Doc 3: 'delivers' (0.4024)  |  The mailman delivers letters every morning
  Doc 4: 'barking' (0.4024)  |  A barking dog rarely bites


### 4.5 — Compare with sklearn

Run the cell below (`[RUN]` — do not modify). Inspect the output and answer the reflection.

In [16]:
# [RUN] — sklearn comparison. Do not modify.
from sklearn.feature_extraction.text import TfidfVectorizer

# sklearn with NO smoothing and NO L2 normalization, to isolate
# the IDF formula difference
tfidf_sk_raw = TfidfVectorizer(
    smooth_idf=False,
    norm=None,
    use_idf=True,
    tokenizer=lambda x: preprocess(x),  # use our own preprocessor
    token_pattern=None,
)
X_sk_raw = tfidf_sk_raw.fit_transform(CORPUS_A).toarray()

print("sklearn TF-IDF (smooth_idf=False, norm=None), doc 0:")
sk_terms = tfidf_sk_raw.get_feature_names_out()
for term, val in sorted(zip(sk_terms, X_sk_raw[0]), key=lambda x: -x[1])[:6]:
    print(f"  '{term}': {val:.4f}")

print("\nYour TF-IDF, doc 0:")
for term, val in sorted(zip(terms, tfidf_A[0]), key=lambda x: -x[1])[:6]:
    print(f"  '{term}': {val:.4f}")

print("\nsklearn IDF formula (when smooth_idf=False): log(N / df(t)) + 1")
print("Classic IDF formula:                          log(N / df(t))")
print("The +1 additive constant is sklearn's bias term to ensure IDF >= 1.")

sklearn TF-IDF (smooth_idf=False, norm=None), doc 0:
  'barked': 2.6094
  'loudly': 2.6094
  'dog': 1.9163
  'mailman': 1.9163
  'barking': 0.0000
  'bites': 0.0000

Your TF-IDF, doc 0:
  'barked': 0.4024
  'loudly': 0.4024
  'dog': 0.2291
  'mailman': 0.2291
  'barking': 0.0000
  'bites': 0.0000

sklearn IDF formula (when smooth_idf=False): log(N / df(t)) + 1
Classic IDF formula:                          log(N / df(t))
The +1 additive constant is sklearn's bias term to ensure IDF >= 1.


### 4.6 — Reflection (manually graded, 6 pts)

**Prompt:** Answer both questions in 4–6 sentences total.

**Question A:** The comparison above shows that sklearn adds +1 to the IDF. What effect does this have on terms with low document frequency vs. high document frequency, and when would you prefer the classic formula?

**Question B:** TF-IDF preserves term order information: True or False? What information does TF-IDF capture that raw BoW counts do not? What information does TF-IDF *lose* compared to the original text that a more powerful representation (e.g. word embeddings) would retain? Give a concrete example of a task where this loss matters.

**Your reflection:**

Adding 1 to every IDF helps high-df terms most in relative terms: in CORPUS_A a df = 1 term like *barked* rises from 1.609 to 2.609 (×1.6) while a df = 2 term like *dog* rises from 0.916 to 1.916 (×2.1), which shrinks the rare-to-common ratio from 1.76 to 1.36, and a term in all 5 documents would get weight 1 instead of 0, so it stays in the vector instead of vanishing. I would prefer the classic formula for ranking documents against a search query without a stop list, because, as J&M note (*Information Retrieval and RAG*), IDF weighting already does much of a stop list's job by down-weighting words that occur in every document, which the classic formula sends to exactly 0 and the +1 lets back in at weight 1. The printed comparison does not isolate the +1, though, because sklearn uses raw counts as TF while this lab divides by document length: *barked* in document 0 scores 1 × 2.609 = 2.6094 in sklearn but 0.25 × 1.609 = 0.4024 here, and I checked that sklearn's matrix equals BoW × (IDF + 1) exactly.

For Question B the answer is False: TF-IDF, like BoW, represents a document as an unordered bag of words, so "the dog bit the mailman" and "the mailman bit the dog" get identical vectors, and what it adds over raw counts is document frequency, which down-weights terms spread across documents (*dog*, df = 2) and promotes distinctive ones, which is why *barked* rather than *dog* is document 0's top term. Beyond word order, it loses any similarity between related words, because each word type gets its own column of the term-document matrix: *dog*, *dogs* and *pets* are orthogonal dimensions, whereas embeddings place them near each other (J&M, *Embeddings*). That loss matters for semantic search or FAQ retrieval, where a user who asks about "pets" should find "The dog barked at the mailman loudly", yet its TF-IDF vector shares no nonzero term with the query, so its cosine similarity is 0.

---

## Submission checklist

Before uploading to Canvas:

- [x] All cells run top-to-bottom without errors (**Runtime → Run all**)
- [x] All four reflection markdown cells contain your written answers (not placeholder text)
- [x] All named output variables defined: `vocab_sizes`, `stemmed_test`, `lemmatized_test`, `vocab_size_no_norm`, `vocab_size_stemmed`, `vocab_size_lemmatized`, `vocab_A`, `bow_A`, `tf_A`, `idf_A`, `tfidf_A`
- [x] `compute_tfidf` uses only NumPy and Python built-ins (no sklearn inside)
- [x] Every function has a complete docstring with args and return value
- [x] File saved as `Lab_3.ipynb`